# 01 — Preprocessing

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
import joblib
import json
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

os.environ['OMP_NUM_THREADS'] = '1'

## 2. Load Data

In [ ]:
props = pd.read_csv("properties.csv")   
cities = pd.read_csv("cities.csv")
districts = pd.read_csv("districts.csv")

print("properties:", props.shape)
print("cities:", cities.shape)
print("districts:", districts.shape)
print()
print("cities.csv columns:", cities.columns.tolist())
print("districts.csv columns:", districts.columns.tolist())

properties: (203874, 26)
cities: (2155, 11)
districts: (25, 5)

cities.csv columns: ['city id', 'district_id', 'name_en', 'name_si', 'name_ta', 'sub_name_en', 'sub_name_si', 'sub_name_ta', 'postcode', 'latitude', 'longitude']
districts.csv columns: ['district id', 'province_id', 'name_en', 'name_si', 'name_ta']


### 3. Data Cleaning

In [3]:
# Filter to apartments for sale only (excludes houses, land, commercial, and rentals)
apt = props[
    (props["type"] == "for_sale") &
    (props["category"].str.contains("Apartment", case=False, na=False))
].copy()

print(f"Apartments for sale: {len(apt)} rows")

Apartments for sale: 5177 rows


In [ ]:
from ml_utils import clean_price, extract_sqft

apt["price_clean"] = clean_price(apt["price"])
apt = apt.dropna(subset=["price_clean"])

apt = apt[apt["price_clean"] > 1_000_000]

print(f"Rows after price cleaning: {len(apt)}")
print(apt["price_clean"].describe())

Rows after price cleaning: 5156
count    5.156000e+03
mean     9.944550e+07
std      1.190022e+09
min      1.200000e+06
25%      3.300000e+07
50%      5.200000e+07
75%      8.000000e+07
max      6.020251e+10
Name: price_clean, dtype: float64


In [5]:
apt["sqft"] = apt["properties"].apply(extract_sqft)
apt = apt.dropna(subset=["sqft"])
apt = apt[apt["sqft"] > 100]

print(f"Rows with valid sqft: {len(apt)}")
print(apt["sqft"].describe())

Rows with valid sqft: 5152
count     5152.000000
mean      1448.112189
std        983.346224
min        167.000000
25%       1050.000000
50%       1280.000000
75%       1600.000000
max      30500.000000
Name: sqft, dtype: float64


## 4. Price per Square Foot

In [ ]:
apt["price_per_sqft"] = apt["price_clean"] / apt["sqft"]

price_per_sqft_before_outliers = apt["price_per_sqft"].copy()

Q1, Q3 = apt["price_per_sqft"].quantile([0.25, 0.75])
IQR = Q3 - Q1
apt = apt[
    (apt["price_per_sqft"] >= Q1 - 1.5 * IQR) &
    (apt["price_per_sqft"] <= Q3 + 1.5 * IQR)
]

print(f"Rows after outlier removal: {len(apt)} (from {len(price_per_sqft_before_outliers)})")
print(apt["price_per_sqft"].describe())

Rows after outlier removal: 4693 (from 5152)
count     4693.000000
mean     39810.954795
std      17457.756938
min        521.059488
25%      28000.000000
50%      36000.000000
75%      48148.148148
max      93270.365998
Name: price_per_sqft, dtype: float64


In [ ]:
apt["city_clean"] = apt["location"].str.strip().str.title()

city_stats = apt.groupby("city_clean").agg(
    avg_price_per_sqft=("price_per_sqft", "mean"),
    listing_count=("price_per_sqft", "count")
).reset_index()

city_stats = city_stats[city_stats["listing_count"] >= 3].reset_index(drop=True)

print(f"Cities with sufficient data: {len(city_stats)}")
city_stats.sort_values("avg_price_per_sqft", ascending=False)

Cities with sufficient data: 44


,city_clean,avg_price_per_sqft,listing_count
9,Colombo 2,62377.565435,397
14,Colombo 7,60353.344350,112
10,Colombo 3,51001.248957,507
12,Colombo 5,50494.040181,559
39,Rajagiriya,44819.545431,544
22,Kandana,44146.841844,3
32,Nawala,40037.420324,27
15,Colombo 8,39239.830731,143
11,Colombo 4,36712.557980,209
35,Nuwara Eliya City,36329.674857,8


In [8]:
apt.to_csv("apt_clean.csv", index=False)
city_stats.to_csv("city_stats_stage1.csv", index=False)
print(f"Saved apt_clean.csv ({len(apt)} rows) and city_stats_stage1.csv ({len(city_stats)} cities)")

Saved apt_clean.csv (4693 rows) and city_stats_stage1.csv (44 cities)
